# Training on Kaggle

My laptop only does about 25 steps/s on CPU, so I run the training here on a Kaggle GPU.

Before running: Settings -> Accelerator: GPU T4, and Internet on. To let it run in the background I use Save Version -> Save & Run All, then download the zips from the Output tab.

In [ ]:
!nvidia-smi

I clone into /tmp so the repo and the .venv don't end up in the notebook output, only the result zips do.

In [ ]:
%cd /tmp
!git clone https://github.com/Samkwizera/deep_q_lerning-group-12.git
%cd /tmp/deep_q_lerning-group-12

In [ ]:
import os

!curl -LsSf https://astral.sh/uv/install.sh | sh
os.environ["PATH"] = "/root/.local/bin:" + os.environ["PATH"]
# kaggle sets this to its inline backend, which the uv venv doesn't have, so matplotlib crashes on import
os.environ["MPLBACKEND"] = "Agg"
!uv sync
# shell commands don't stop the notebook when they fail, so check by hand
if _exit_code != 0:
    raise RuntimeError("uv sync failed, check the output above")
!uv run python -c "import torch; print(torch.__version__, torch.cuda.is_available())"

## Runs

The baseline (my exp01) is done: default settings for 500k steps, about 22 minutes on a T4. It only started learning around 200k steps, so 500k is the budget for all experiments.

Now running exp02 to exp09, where I only change the epsilon settings. exp10 comes after, once I know which values worked best.

In [ ]:
import shutil

common = "--total-steps 500000"

runs = {
    "Ihimbazwe_Samuel/exp02": "--eps-start 1.0 --eps-end 0.01 --eps-decay 100000",
    "Ihimbazwe_Samuel/exp03": "--eps-start 1.0 --eps-end 0.1 --eps-decay 100000",
    "Ihimbazwe_Samuel/exp04": "--eps-start 1.0 --eps-end 0.2 --eps-decay 100000",
    "Ihimbazwe_Samuel/exp05": "--eps-start 1.0 --eps-end 0.05 --eps-decay 25000",
    "Ihimbazwe_Samuel/exp06": "--eps-start 1.0 --eps-end 0.05 --eps-decay 250000",
    "Ihimbazwe_Samuel/exp07": "--eps-start 1.0 --eps-end 0.05 --eps-decay 400000",
    "Ihimbazwe_Samuel/exp08": "--eps-start 0.5 --eps-end 0.05 --eps-decay 100000",
    "Ihimbazwe_Samuel/exp09": "--eps-start 0.1 --eps-end 0.01 --eps-decay 50000",
}

# -u so the episode logs show up live instead of all at the end
for out_dir, run_args in runs.items():
    !uv run python -u train.py {common} {run_args} --out-dir {out_dir}
    if _exit_code != 0:
        raise RuntimeError(f"{out_dir} failed, check the error above")
    # zip straight away so finished runs are saved even if a later one crashes
    shutil.make_archive(f"/kaggle/working/{out_dir.replace('/', '_')}", "zip", out_dir)

In [ ]:
from IPython.display import Image, display

for out_dir in runs:
    print(out_dir)
    display(Image(f"{out_dir}/plot.png"))

In [ ]:
!ls -lh /kaggle/working
